In [0]:
BASE_DIR = '/Volumes/sales/landing/volumes/OrderItems2/'
CHECKPOINT_DIR = '/Volumes/sales/landing/volumes/checkpoint_dir'
CHECKPOINT_LOCATION = f'{CHECKPOINT_DIR}/OrderItems'
CATALOG = 'SALES'
SCHEMA = 'BRONZE'
TARGET_TABLE = 'ORDERITEMS'
print(f'Checkpoint Location: {CHECKPOINT_LOCATION}')
print(f'Base Location: {BASE_DIR}')

In [0]:
orderitems_schema = """order_id STRING, 
                 order_item_id INT, 
                 product_id STRING, 
                 seller_id STRING, 
                 shipping_limit_date TIMESTAMP,
                 price DECIMAL(10,2),
                 freght_value DECIMAL(10,2)
                 """

In [0]:
orderitem_df = (spark.read.format('csv').option('header','true').load(f'{BASE_DIR}')
           )
display(orderitem_df.limit(3))
orderitem_df.printSchema()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

cust_df = (
            spark.readStream.format('cloudFiles')
                 .option('cloudFiles.format', 'csv')
                 .option('cloudFiles.schemaEvolutionMode', 'rescue')
                 .option('header', 'true')
                 .schema(orderitems_schema)
                 .load(f'{BASE_DIR}')
        )
cust_trans_df = cust_df.select(
    "order_id", 
                 order_item_id INT, 
                 product_id STRING, 
                 seller_id STRING, 
                 shipping_limit_date TIMESTAMP,
                 price DECIMAL(10,2),
                 freght_value DECIMAL(10,2),
    "_metadata.file_name",
    "_metadata.file_path",).withColumn('load_timestamp', current_timestamp())
